In [ ]:
from osgeo import gdal, osr
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import sobel, uniform_filter, generic_filter
import os

gdal.PushErrorHandler('CPLQuietErrorHandler')

# ============================================================================
# CONFIGURATION
# ============================================================================

output_dir = r"E:\output\terrain_analysis"
os.makedirs(output_dir, exist_ok=True)

ice_prob_east = r"E:\output\ice_analysis\ice_probability_east.tif"
ice_prob_west = r"E:\output\ice_analysis\ice_probability_west.tif"

print("="*70)
print("WEEK 2: TERRAIN ANALYSIS PIPELINE")
print("="*70)

# ============================================================================
# STEP 1: LOAD ICE PROBABILITY MAPS
# ============================================================================

print("\nSTEP 1: Loading ice probability maps...")
print("-"*70)

ice_ds_east = gdal.Open(ice_prob_east)
ice_ds_west = gdal.Open(ice_prob_west)

ice_east = ice_ds_east.GetRasterBand(1).ReadAsArray().astype(np.float32)
ice_west = ice_ds_west.GetRasterBand(1).ReadAsArray().astype(np.float32)

print(f"EAST ice map: {ice_east.shape}")
print(f"WEST ice map: {ice_west.shape}")

# ============================================================================
# STEP 2: CREATE SYNTHETIC LUNAR DEM (FAST)
# ============================================================================

print("\nSTEP 2: Creating synthetic lunar DEM (fast method)...")
print("-"*70)

shape_east = ice_east.shape
shape_west = ice_west.shape

def create_dem_fast(shape):
    """Create DEM without crater loop (much faster)."""
    rows, cols = shape
    dem = np.full(shape, -3000.0, dtype=np.float32)
    
    # Just add simple slope + noise (no craters)
    yy, xx = np.meshgrid(np.linspace(0, 150, cols), np.linspace(0, 100, rows))
    dem += 0.5 * (xx + yy)
    dem += np.random.normal(0, 8, shape)
    
    return dem

dem_east = create_dem_fast(shape_east)
dem_west = create_dem_fast(shape_west)

print(f"EAST DEM: {dem_east.min():.0f} to {dem_east.max():.0f} m")
print(f"WEST DEM: {dem_west.min():.0f} to {dem_west.max():.0f} m")

# ============================================================================
# STEP 3: COMPUTE SLOPE (DOWNSAMPLED)
# ============================================================================

print("\nSTEP 3: Computing slope (downsampled)...")
print("-"*70)

def compute_slope_fast(dem, pixel_size_m=200):
    """Fast slope using scipy sobel."""
    dz_dx = sobel(dem.astype(float), axis=1) / (2 * pixel_size_m)
    dz_dy = sobel(dem.astype(float), axis=0) / (2 * pixel_size_m)
    slope_deg = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))
    return slope_deg

slope_east = compute_slope_fast(dem_east)
slope_west = compute_slope_fast(dem_west)

print(f"EAST slope: {np.nanmean(slope_east):.2f}° (mean)")
print(f"WEST slope: {np.nanmean(slope_west):.2f}° (mean)")

# ============================================================================
# STEP 4: COMPUTE ROUGHNESS (FAST)
# ============================================================================

print("\nSTEP 4: Computing roughness...")
print("-"*70)

def compute_roughness_fast(dem, window_size=5):
    """Fast roughness using uniform_filter."""
    roughness = generic_filter(dem.astype(float), np.std, size=window_size)
    return roughness

roughness_east = compute_roughness_fast(dem_east)
roughness_west = compute_roughness_fast(dem_west)

print(f"EAST roughness: {np.nanmean(roughness_east):.2f}m (mean)")
print(f"WEST roughness: {np.nanmean(roughness_west):.2f}m (mean)")

# ============================================================================
# STEP 5: COMPUTE PSR
# ============================================================================

print("\nSTEP 5: Computing PSR...")
print("-"*70)

def compute_psr_fast(dem, relief_threshold=120):
    """Fast PSR using uniform_filter."""
    local_max = uniform_filter(dem.astype(float), size=25, mode='constant', cval=dem.max())
    relief = local_max - dem
    psr = (relief > relief_threshold).astype(np.float32)
    return psr

psr_east = compute_psr_fast(dem_east)
psr_west = compute_psr_fast(dem_west)

print(f"EAST PSR: {100*np.mean(psr_east):.2f}%")
print(f"WEST PSR: {100*np.mean(psr_west):.2f}%")

# ============================================================================
# STEP 6: LANDING SITE SCORING
# ============================================================================

print("\nSTEP 6: Scoring landing sites...")
print("-"*70)

def score_landing_sites(ice, slope, roughness, psr):
    """Fast landing site scoring."""
    slope_score = np.clip(1.0 - (slope / 15.0), 0, 1)
    roughness_score = np.clip(1.0 - (roughness / 10.0), 0, 1)
    score = 0.35*ice + 0.30*slope_score + 0.20*roughness_score + 0.15*psr
    return np.clip(score, 0, 1)

landing_east = score_landing_sites(ice_east, slope_east, roughness_east, psr_east)
landing_west = score_landing_sites(ice_west, slope_west, roughness_west, psr_west)

best_pos_e = np.unravel_index(np.nanargmax(landing_east), landing_east.shape)
best_pos_w = np.unravel_index(np.nanargmax(landing_west), landing_west.shape)
best_score_e = landing_east[best_pos_e]
best_score_w = landing_west[best_pos_w]

print(f"EAST best: ({best_pos_e[0]}, {best_pos_e[1]}) → Score: {best_score_e:.3f}")
print(f"WEST best: ({best_pos_w[0]}, {best_pos_w[1]}) → Score: {best_score_w:.3f}")

# ============================================================================
# STEP 7: SAVE ALL OUTPUTS (PARALLEL)
# ============================================================================

print("\nSTEP 7: Saving outputs...")
print("-"*70)

outputs = [
    ("dem", [dem_east, dem_west]),
    ("slope", [slope_east, slope_west]),
    ("roughness", [roughness_east, roughness_west]),
    ("psr", [psr_east, psr_west]),
    ("landing_site_score", [landing_east, landing_west]),
]

driver = gdal.GetDriverByName('GTiff')
for name, [data_e, data_w] in outputs:
    for swath, data in [("east", data_e), ("west", data_w)]:
        path = os.path.join(output_dir, f'{name}_{swath}.tif')
        ds = driver.Create(path, data.shape[1], data.shape[0], 1, gdal.GDT_Float32)
        ds.GetRasterBand(1).WriteArray(data)
        ds.FlushCache()
        del ds
    print(f"✓ Saved: {name}_*.tif")

# ============================================================================
# STEP 8: SIMPLE VISUALIZATION (FAST)
# ============================================================================

print("\nSTEP 8: Creating visualizations...")
print("-"*70)

for swath_name, landing in [("east", landing_east), ("west", landing_west)]:
    fig, ax = plt.subplots(figsize=(14, 10))
    im = ax.imshow(landing, cmap='RdYlGn', vmin=0, vmax=1)
    best_pos = np.unravel_index(np.nanargmax(landing), landing.shape)
    ax.plot(best_pos[1], best_pos[0], 'b*', markersize=30, markeredgecolor='white', markeredgewidth=2)
    ax.set_title(f'Landing Site Score — {swath_name.upper()}', fontsize=16, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Score')
    
    path = os.path.join(output_dir, f'landing_site_score_{swath_name}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: landing_site_score_{swath_name}.png")
    plt.close()

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*70)
print("WEEK 2 — COMPLETE")
print("="*70)
print(f"""
✓ Terrain analysis complete
✓ Landing sites identified
✓ All GeoTIFF files saved

BEST SITES:
  EAST: ({best_pos_e[0]}, {best_pos_e[1]}) Score={best_score_e:.3f}
  WEST: ({best_pos_w[0]}, {best_pos_w[1]}) Score={best_score_w:.3f}

NEXT: Week 3 — Rover Path Planning
""")
print("="*70)

WEEK 2: TERRAIN ANALYSIS PIPELINE

STEP 1: Loading ice probability maps...
----------------------------------------------------------------------


e:\conda_envs\moon_ice_mapping\Lib\site-packages\osgeo\gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


EAST ice map: (12237, 12794)
WEST ice map: (24794, 24181)

STEP 2: Creating synthetic lunar DEM (fast method)...
----------------------------------------------------------------------
EAST DEM: -3035 to -2839 m
WEST DEM: -3036 to -2837 m

STEP 3: Computing slope (downsampled)...
----------------------------------------------------------------------
EAST slope: 4.95° (mean)
WEST slope: 4.95° (mean)

STEP 4: Computing roughness...
----------------------------------------------------------------------
